In [1]:
import torch
import torch.nn as nn
import json
import gym
import gym_cityflow
import cityflow
import matplotlib.pyplot as plt
import os
import torch.nn.functional as F
import numpy as np
import torch.optim as optim
from torch_geometric.nn import GATConv
from torch_geometric.utils import add_self_loops

In [2]:
class DuelingGAT(torch.nn.Module):
    def __init__(
        self,
        in_dim: int,      
        hidden_dim: int,  
        action_dim: int,    
        heads: int = 4      
    ):
        super().__init__()
        self.gat1 = GATConv(in_dim, hidden_dim, heads=heads, concat=True)
        self.gat2 = GATConv(hidden_dim * heads, hidden_dim, heads=1, concat=False)

        self.val_fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, 1)
        )
        
        self.adv_fc = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim // 2, action_dim)
        )

    def forward(self, x: torch.Tensor, edge_index: torch.LongTensor): 
        h = F.elu(self.gat1(x, edge_index))
        h = F.elu(self.gat2(h, edge_index))
        V = self.val_fc(h)                        
        A = self.adv_fc(h)                        
        Q = V + (A - A.mean(dim=1, keepdim=True)) 

        return Q


In [3]:
CONFIG_PATH = "Intersections_4/sample_config.json"
CHECKPOINT = "checkpoints/qnet_gat__ep250.pth"
NUM_AGENTS = 16
OBS_DIM    = 72
ACTION_DIM = 9
STEPS      = 1000
HIDDEN_DIM   = 64
HEADS        = 4

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DuelingGAT(OBS_DIM, HIDDEN_DIM, ACTION_DIM, heads=HEADS).to(device)
state_dict = torch.load(CHECKPOINT, map_location=device)
model.load_state_dict(state_dict)
model.eval()

DuelingGAT(
  (gat1): GATConv(72, 64, heads=4)
  (gat2): GATConv(256, 64, heads=1)
  (val_fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=32, out_features=1, bias=True)
  )
  (adv_fc): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU(inplace=True)
    (2): Linear(in_features=32, out_features=9, bias=True)
  )
)

In [5]:
cfg     = json.load(open(CONFIG_PATH))
roadnet = json.load(open(cfg["dir"] + cfg["roadnetFile"]))
nodes   = [i["id"] for i in roadnet["intersections"] if not i.get("virtual",False)]
nodes   = sorted(nodes)
node2idx = {n:idx for idx,n in enumerate(nodes)}

edges = []
for road in roadnet["roads"]:
    u,v = road.get("startIntersection"), road.get("endIntersection")
    if u in node2idx and v in node2idx:
        ui,vi = node2idx[u], node2idx[v]
        edges += [[ui,vi],[vi,ui]]
edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
edge_index, _ = add_self_loops(edge_index, num_nodes=len(nodes))
edge_index = edge_index.to(device)

In [6]:
env = gym.make(
    id="cityflow-v0",
    configPath=CONFIG_PATH,
    episodeSteps=STEPS
)
raw_obs = env.reset()
intersection_ids = sorted(raw_obs.keys())
assert len(intersection_ids) == NUM_AGENTS

/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:174: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed a `seed` instead of using `Env.seed` for resetting the environment random number generator.
  logger.warn(
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:187: UserWarning: WARN: Future gym versions will require that `Env.reset` can be passed `options` to allow the environment initialisation to be passed additional information.
  logger.warn(
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:195: UserWarning: WARN: The result returned by `env.reset()` was not a tuple of the form `(obs, info)`, where `obs` is a observation and `info` is a dictionary containing additional information. Actual type: `<class 'dict'>`
  logger.warn(


In [7]:
total_reward      = 0.0
step_rewards      = []
queue_lengths     = []
prev_vehicles     = set(env.eng.get_vehicles(include_waiting=True))
arrived_vehicles  = set()

lane_wait = env.eng.get_lane_waiting_vehicle_count()
queue_lengths.append(sum(lane_wait.values()))

In [8]:
obs = [ np.array(raw_obs[i],dtype=np.float32).flatten() for i in intersection_ids ]

for t in range(1, STEPS+1):
    x = torch.tensor(obs, dtype=torch.float32, device=device)    
    with torch.no_grad():
        qv = model(x, edge_index)                                 
    actions = qv.argmax(dim=1).cpu().tolist()                     
    next_raw_obs, rewards, done, _ = env.step(actions)
    step_r = sum(r for _,r in rewards)
    total_reward += step_r
    step_rewards.append(step_r)

    cur_vehicles = set(env.eng.get_vehicles(include_waiting=True))
    new_arrs     = prev_vehicles - cur_vehicles
    arrived_vehicles |= new_arrs
    prev_vehicles = cur_vehicles

    lw = env.eng.get_lane_waiting_vehicle_count()
    queue_lengths.append(sum(lw.values()))
    obs = [ np.array(next_raw_obs[i],dtype=np.float32).flatten()
            for i in intersection_ids ]

    print(f"Step {t:4d} | Reward {step_r:8.1f} | Queue {queue_lengths[-1]:4d}")
    if done:
        print(f"Env signaled done at step {t}")
        break

num_arrived      = len(arrived_vehicles)
total_wait_time  = sum(queue_lengths)                   
avg_wait_per_car = total_wait_time / num_arrived        if num_arrived>0 else float("nan")
avg_travel_time  = env.eng.get_average_travel_time()    
active_vehicles  = env.eng.get_vehicle_count()

print("\n===== Dueling GAT-DQN Evaluation =====")
print(f"Total steps executed       : {t}")
print(f"Total reward accumulated   : {total_reward:.1f}")
print(f"Vehicles completed routes  : {num_arrived}")
print(f"Active vehicles remaining  : {active_vehicles}")
print(f"Avg waiting time per car   : {avg_wait_per_car:.2f} s")
print(f"Avg travel time (all)      : {avg_travel_time:.2f} s")


/scratch/19527425/ipykernel_469211/3276665258.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  x = torch.tensor(obs, dtype=torch.float32, device=device)    # [N,obs_dim]
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:219: DeprecationWarning: WARN: Core environment is written in old step API which returns one bool instead of two. It is recommended to rewrite the environment with new step API. 
  logger.deprecation(
/user/gaganvad/.local/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:146: UserWarning: WARN: The obs returned by the `step()` method was expecting a numpy array, actual type: <class 'list'>
  logger.warn(f"{pre} was expecting a numpy array, actual type: {type(obs)}")
/user/gaganvad/.local/lib/python3.10/si

Step    1 | Reward      0.0 | Queue    0
Step    2 | Reward      0.0 | Queue    0
Step    3 | Reward      0.0 | Queue    0
Step    4 | Reward      0.0 | Queue    0
Step    5 | Reward      0.0 | Queue    0
Step    6 | Reward      0.0 | Queue    0
Step    7 | Reward      0.0 | Queue    0
Step    8 | Reward      0.0 | Queue    0
Step    9 | Reward      0.0 | Queue    0
Step   10 | Reward      0.0 | Queue    0
Step   11 | Reward      0.0 | Queue    0
Step   12 | Reward      0.0 | Queue    0
Step   13 | Reward      0.0 | Queue    0
Step   14 | Reward      0.0 | Queue    0
Step   15 | Reward      0.0 | Queue    0
Step   16 | Reward      0.0 | Queue    0
Step   17 | Reward      0.0 | Queue    0
Step   18 | Reward      0.0 | Queue    0
Step   19 | Reward      0.0 | Queue    0
Step   20 | Reward      0.0 | Queue    0
Step   21 | Reward      0.0 | Queue    0
Step   22 | Reward      0.0 | Queue    0
Step   23 | Reward      0.0 | Queue    0
Step   24 | Reward      0.0 | Queue    0
Step   25 | Rewa